### Breast cancer detection

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [4]:
df = pd.read_csv('./data/breast-cancer.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [5]:
df.drop(columns=['id'],inplace=True)

In [6]:
X = df.iloc[:,1:]
y = df.iloc[:,0]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [8]:
scaler = StandardScaler()
encoder = LabelEncoder()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [9]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test) 

In [10]:
X_train_tensor.shape

torch.Size([455, 30])

In [19]:
class MySimpleNN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.linear = nn.Linear(num_features,1)
        self.sigmoid = nn.Sigmoid()

    def forward(self,features):
        features = features.float()
        out = self.linear(features)
        out = self.sigmoid(out)
        return out

In [20]:
class CustomDataset(Dataset):
    def __init__(self,feature,label):
        self.feature = feature
        self.label = label

    def __len__(self):
        return len(self.feature)

    def __getitem__(self, index):
        return self.feature[index], self.label[index]

In [21]:
train_dataset = CustomDataset(X_train_tensor,y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)


In [22]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [23]:
learning_rate = 0.01
epochs = 30


In [24]:
loss_function = nn.BCELoss()

In [25]:
model = MySimpleNN(X_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

for epoch in range(0,epochs):
    for batch_feature, batch_label in train_loader:
        y_pred = model(batch_feature)
        loss = loss_function(y_pred, batch_label.view(-1,1).float())

        loss.backward()

        optimizer.step() 
        optimizer.zero_grad()

        print(f"epochs {epoch + 1}, loss: {loss.item()}")

epochs 1, loss: 0.6831672191619873
epochs 1, loss: 0.6936541795730591
epochs 1, loss: 0.5589395761489868
epochs 1, loss: 0.543997049331665
epochs 1, loss: 0.6229898929595947
epochs 1, loss: 0.5758953094482422
epochs 1, loss: 0.5631861090660095
epochs 1, loss: 0.5954672694206238
epochs 1, loss: 0.4767342507839203
epochs 1, loss: 0.4961695671081543
epochs 1, loss: 0.5199646949768066
epochs 1, loss: 0.5275059938430786
epochs 1, loss: 0.5578632354736328
epochs 1, loss: 0.5588252544403076
epochs 1, loss: 0.4215209484100342
epochs 2, loss: 0.5005429983139038
epochs 2, loss: 0.4353872239589691
epochs 2, loss: 0.5026835203170776
epochs 2, loss: 0.5588550567626953
epochs 2, loss: 0.45418214797973633
epochs 2, loss: 0.3875160217285156
epochs 2, loss: 0.4547383487224579
epochs 2, loss: 0.4029897153377533
epochs 2, loss: 0.45684346556663513
epochs 2, loss: 0.4712570905685425
epochs 2, loss: 0.4447014331817627
epochs 2, loss: 0.4323568046092987
epochs 2, loss: 0.5129537582397461
epochs 2, loss: 0.3

In [26]:
# model evaluation 
correct = 0
total = 0

with torch.no_grad():
    for batch_feature, batch_label in test_loader:
        y_pred = model(batch_feature)
        y_pred = (y_pred > 0.5).float()

        # Count correct predictions
        correct += (y_pred == batch_label.view(-1,1)).sum().item()
        total += batch_label.size(0)

accuracy = correct / total
print("Model accuracy:", accuracy)

Model accuracy: 0.9736842105263158


In [ ]:
""